# Customer Graph With Segmentation
This notebook creates a graph from a sample of the main [customer experience demo](https://neo4j.com/developer/demos/cx-demo/). It focuses on a subset of the data model that describes customer product ordering behavior. Data is loaded from an `orders.csv` file, after which graph analytics algorithms are used to create customer segments based on purchasing behavior. These segments are then labeled with titles and descriptions based on purchase patterns using an LLM workflow. The segmentation will be used in the MCP tooling which is configured in the last step.

 __Note that you will need an OpenAI api key to run this notebook__

 ![](img/subset-graph-model.png)

In [57]:
#get env setup
import getpass
import os
from dotenv import load_dotenv

load_dotenv('cx.env', override=True)

if not os.environ.get('NEO4J_URI'):
    os.environ['NEO4J_URI'] = getpass.getpass('NEO4J_URI:\n')
if not os.environ.get('NEO4J_USERNAME'):
    os.environ['NEO4J_USERNAME'] = getpass.getpass('NEO4J_USERNAME:\n')
if not os.environ.get('NEO4J_PASSWORD'):
    os.environ['NEO4J_PASSWORD'] = getpass.getpass('NEO4J_PASSWORD:\n')
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY:\n')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

## Create Graph From CSV

In [59]:
from neo4j import GraphDatabase

#instantiate driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# create constraints (for efficient loading and queries)
driver.execute_query('CREATE CONSTRAINT IF NOT EXISTS FOR (n:Customer) REQUIRE (n.customerId) IS NODE KEY')
driver.execute_query('CREATE CONSTRAINT IF NOT EXISTS FOR (n:OrderLineItem) REQUIRE (n.invoiceLineId) IS NODE KEY')
driver.execute_query('CREATE CONSTRAINT IF NOT EXISTS FOR (n:Order) REQUIRE (n.invoiceNo) IS NODE KEY')
driver.execute_query('CREATE CONSTRAINT IF NOT EXISTS FOR (n:Product) REQUIRE (n.stockCode) IS NODE KEY')

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x1189b5b20>, keys=[])

In [60]:
# helper function for batch loading
def chunks(xs, n=10_000):
    n = max(1, n)
    return [xs[i:i + n] for i in range(0, len(xs), n)]

In [66]:
# load graph from csv
from tqdm import tqdm
import pandas as pd

order_df = pd.read_csv('orders.csv')
order_df['invoiceDate'] = pd.to_datetime(order_df['invoiceDate']) #convert to datetime

for records in tqdm(chunks(order_df.to_dict(orient='records')), desc="Loading data to Neo4j"):
    driver.execute_query('''
        UNWIND $records as rec
        // merge nodes
        MERGE (c:Customer {customerId: rec.customerId})
        MERGE (o:Order {invoiceNo: rec.invoiceNo})
        MERGE (i:OrderLineItem {invoiceLineId: rec.invoiceLineId})
        MERGE (p:Product {stockCode: rec.stockCode})
        SET c.firstName = rec.firstName, c.lastName = rec.lastName,
            o.invoiceDate = rec.invoiceDate,
            i.quantity = rec.quantity, i.unitPrice = rec.unitPrice,
            p.description = rec.description

        // merge relationships
        MERGE (c)-[:ORDERED]->(o)
        MERGE (o)-[:LINE_ITEM]->(i)
        MERGE (i)-[:PRODUCT]->(p)
    ''', records = records)

Loading data to Neo4j: 100%|██████████| 40/40 [00:34<00:00,  1.16it/s]



## Segmentation by Purchasing Behavior

The following uses Graph Analytics via the [Graph Data Science library](https://neo4j.com/docs/graph-data-science/current/) to create segments based on purchasing behavior. It does this by:

1. **Staging a Projection**: Creating a [Graph Projection](https://neo4j.com/docs/graph-data-science/current/management-ops/graph-creation/graph-project-cypher-projection/) which is an in-memory version of the graph for running algorithms efficiently. In this case we condense the data model to focus on the `(Customer)-[PURCHASED]->(Product)` pattern. This type of graph, where we have a directed relationship between two node types, is often referred to as a [bipartite graph](https://en.wikipedia.org/wiki/Bipartite_graph).

2. **Identifying Similar Customers**: Find customers with similar purchases and connect them with `SIMILAR_PURCHASES_TO` relationships. This is accomplished by running [FastRP node embeddings](https://neo4j.com/docs/graph-data-science/current/machine-learning/node-embeddings/fastrp/) followed by the [k-nearest neighbor (KNN) algorithm](https://neo4j.com/docs/graph-data-science/current/algorithms/knn/). This allows us to identify pairs of customers that are close together in the purchasing graph and draw `SIMILAR_PURCHASES_TO` relationships between them in an efficient way - using statistical estimation instead of an `O(n^2)` aggregation on purchase count which doesn't scale well.

3. **Create Customer Communities**: Run the [Louvain algorithm](https://neo4j.com/docs/graph-data-science/current/algorithms/louvain/) to create "communities" (clusters based on relationships) by optimizing modularity - basically maximizing `(Customer)-[SIMILAR_PURCHASES_TO]-(Customer)` connections within communities while minimizing such connections between different communities. These communities can then be interpreted as behavioral customer segments.


In [22]:
from neo4j import GraphDatabase

#instantiate driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# clear previous graph if exists
driver.execute_query("""
CALL gds.graph.drop('weighted-purchases', false)
YIELD graphName
""")

# projection
driver.execute_query("""
MATCH (c:Customer)-[:ORDERED]->(o:Order)-[:LINE_ITEM]->(item:OrderLineItem)-[:PRODUCT]->(p)
WHERE NOT o:Return
WITH c, sum(item.quantity) AS totalQuantity, collect(item) AS items
UNWIND items AS item
MATCH (item)-[:PRODUCT]->(p)
WITH c, p, toFloat(item.quantity)/totalQuantity AS purchaseProportion
WITH gds.graph.project('weighted-purchases', c, p, {
    sourceNodeLabels: labels(c),
    targetNodeLabels: labels(p),
    relationshipType: "PURCHASED",
    relationshipProperties: { purchaseProportion: purchaseProportion}},
    {undirectedRelationshipTypes: ['PURCHASED']}) AS g
RETURN g.graphName AS graph, g.nodeCount AS nodes, g.relationshipCount AS rels
""")

# embeddings
driver.execute_query("""
CALL gds.fastRP.mutate('weighted-purchases', {
    embeddingDimension: 256,
    randomSeed: 7474,
    relationshipWeightProperty: 'purchaseProportion',
    mutateProperty: 'purchaseEmbedding'
})
""")

# KNN
driver.execute_query("""
CALL gds.knn.mutate('weighted-purchases', {
  nodeLabels: ['Customer'],
  nodeProperties:['purchaseEmbedding'],
  mutateRelationshipType:'SIMILAR_PURCHASES_TO',
  mutateProperty:'score',
  sampleRate: 1.0,
  initialSampler: 'randomWalk',
  concurrency: 1,
  similarityCutoff: 0.6
})
""")

# louvain communities - segment ids written back to the DB
driver.execute_query("""
CALL gds.louvain.write('weighted-purchases', {
  nodeLabels: ['Customer'],
  relationshipTypes:['SIMILAR_PURCHASES_TO'],
  relationshipWeightProperty: 'score',
  writeProperty: 'segmentId',
  concurrency:1})
""")



EagerResult(records=[<Record modularity=0.8260642554963326 modularities=[0.7921260140278896, 0.8260642554963326] ranLevels=2 communityCount=51 communityDistribution={'min': 1, 'p5': 1, 'max': 372, 'p999': 372, 'p99': 372, 'p1': 1, 'p10': 2, 'p90': 168, 'p50': 68, 'p25': 28, 'p75': 120, 'p95': 214, 'mean': 85.07843137254902} preProcessingMillis=0 computeMillis=555 postProcessingMillis=3 writeMillis=208 nodePropertiesWritten=4339 configuration={'writeProperty': 'segmentId', 'jobId': '64ec1dfa-9ec7-4c80-a7f9-99948f55be00', 'sudo': False, 'maxIterations': 10, 'maxLevels': 10, 'seedProperty': None, 'writeToResultStore': False, 'writeConcurrency': 1, 'relationshipWeightProperty': 'score', 'logProgress': True, 'nodeLabels': ['Customer'], 'concurrency': 1, 'includeIntermediateCommunities': False, 'relationshipTypes': ['SIMILAR_PURCHASES_TO'], 'tolerance': 0.0001, 'consecutiveIds': False}>], summary=<neo4j._work.summary.ResultSummary object at 0x1100c5cc0>, keys=['modularity', 'modularities', '



## Labeling Segments

Once we have customer segments, we can use LLM workflows to summarize them and enrich the graph. In this case we will keep it very simple:

1. **Create Purchase Summaries**: Use Cypher queries to create purchase summaries by segment.

2. **Generate Labels**: Create an LLM workflow for labeling each segment based on that summary - a short title and description.

3. **Update the Graph**: Write the title and description back to the graph. If we wanted to be more formal we could create dedicated `Segment` nodes for this, but instead I will write the title and description fields to `Customer` properties (keeping the data model simpler).

In [31]:
# pull purchase summaries by segment
segment_purchase_details = driver.execute_query("""
MATCH (c:Customer)
WITH c.segmentId AS segmentId,
  count(c) AS numberOfCustomersInSegment
WHERE numberOfCustomersInSegment > 1
WITH segmentId, numberOfCustomersInSegment
MATCH (c {segmentId:segmentId})-[:ORDERED]->(o:Order)-[:LINE_ITEM]->(item:OrderLineItem)-[:PRODUCT]->(p)
WHERE NOT o:Return
WITH numberOfCustomersInSegment,
  c.segmentId AS segmentId,
  p.stockCode AS stockCode,
  p.description AS description,
  count(*) AS productPurchaseCount
RETURN segmentId,
  numberOfCustomersInSegment,
  collect({stockCode:stockCode,
    description:description,
    purchaseCount:productPurchaseCount}) AS productPurchaseDetails
""", result_transformer_= lambda r: r.data())
segment_purchase_details[:1]

[{'segmentId': 3729,
  'numberOfCustomersInSegment': 84,
  'productPurchaseDetails': [{'purchaseCount': 35,
    'stockCode': '85123A',
    'description': 'CREAM HANGING HEART T-LIGHT HOLDER'},
   {'purchaseCount': 19,
    'stockCode': '71053',
    'description': 'WHITE MOROCCAN METAL LANTERN'},
   {'purchaseCount': 18,
    'stockCode': '84406B',
    'description': 'CREAM CUPID HEARTS COAT HANGER'},
   {'purchaseCount': 29,
    'stockCode': '84029G',
    'description': 'KNITTED UNION FLAG HOT WATER BOTTLE'},
   {'purchaseCount': 27,
    'stockCode': '84029E',
    'description': 'RED WOOLLY HOTTIE WHITE HEART.'},
   {'purchaseCount': 23,
    'stockCode': '22752',
    'description': 'SET 7 BABUSHKA NESTING BOXES'},
   {'purchaseCount': 17,
    'stockCode': '21730',
    'description': 'GLASS STAR FROSTED T-LIGHT HOLDER'},
   {'purchaseCount': 24,
    'stockCode': '22633',
    'description': 'HAND WARMER UNION JACK'},
   {'purchaseCount': 23,
    'stockCode': '22632',
    'description': 'HA

In [50]:
# helper functions and classes for running LLM flow with structured segment description outputs
import asyncio
from typing import List
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from tqdm.asyncio import tqdm as tqdm_async

class SegmentDescription(BaseModel):
    segmentId: int = Field(..., description="segment id as provided")
    title: str = Field(..., description="Short title of customer segment based on purchases. Weight more frequent purchases higher")
    description: str = Field(..., description="Short description of customer segment based on purchases. 1-2 sentences max. Weight more frequent purchases higher")

def chunks(xs, n=1_000):
    n = max(1, n)
    return [xs[i:i + n] for i in range(0, len(xs), n)]

class TextExtractor:
    def __init__(self,
                 llm_with_struct_output,
                 prompt_template: PromptTemplate):
        self.llm = llm_with_struct_output
        self.prompt_template = prompt_template

    async def extract(self, texts: List[str], semaphore) -> BaseModel:
        async with semaphore:
            prompt = self.prompt_template.invoke({'texts': '\n\n'.join(texts)})
            # Use structured LLM for extraction
            entity: BaseModel = await self.llm.ainvoke(prompt)
        return entity


    async def extract_all(self, texts: List[str], chunk_size=1, max_workers=10) -> List[BaseModel]:
        # Create a semaphore with the desired number of workers
        semaphore = asyncio.Semaphore(max_workers)

        # Create tasks with the semaphore
        text_chunks = chunks(texts, chunk_size)
        tasks = [self.extract(text_chunk, semaphore) for text_chunk in text_chunks]

        # Explicitly update progress using `tqdm` as tasks complete
        entities: List[BaseModel] = []
        with tqdm_async(total=len(tasks), desc="extracting texts") as pbar:
            for future in asyncio.as_completed(tasks):
                result = await future
                entities.append(result)
                pbar.update(1)  # Increment progress bar for each completed task
        return entities

In [51]:
# Run LLM workflow for creating segment titles and descriptions
from langchain_openai import ChatOpenAI
import json

#Get LLM api key
load_dotenv('cx.env', override=True)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI AI API key: ")

# Define Prompt and LLM with structured output
prompt_template = PromptTemplate.from_template("""
Describe the Customer Segment based on purchase behavior.
The Retail store focuses on home decore, accessories, and vintage items so you don't need to describe that. Get more specific so we can differentiate between segments.

# Purchase Behavior
{texts}
""")
llm = ChatOpenAI(model="gpt-4.1", temperature=0).with_structured_output(SegmentDescription)

# Perform inference
text_extractor = TextExtractor(llm, prompt_template)
segment_descriptions = await text_extractor.extract_all([json.dumps(s, indent=2) for s in segment_purchase_details])

extracting texts: 100%|██████████| 46/46 [00:55<00:00,  1.21s/it]


In [52]:
#visualize output
segment_descriptions[:3]

[SegmentDescription(segmentId=3587, title="Children's Craft & Lunchware Enthusiasts", description="This segment is characterized by frequent purchases of children's craft kits (especially feltcraft dolls, cushions, and kits), lunch bags with playful designs, and children's cutlery sets. Customers in this group are likely parents or gift-givers focused on creative, playful, and practical items for children."),
 SegmentDescription(segmentId=3275, title='Family & Kids Party Planners', description="This segment frequently purchases children's lunch bags, cake cases, party supplies, and themed kitchenware, especially with polka dot, retro, and character designs. They are likely parents or gift buyers who focus on organizing kids' parties and family gatherings, with a strong interest in playful, colorful, and practical items for children and home entertaining."),
 SegmentDescription(segmentId=4022, title='Family & Kids Gift Givers', description="This segment frequently purchases children's l

In [56]:
# write to segment title and description back to the graph
segment_purchase_details = driver.execute_query("""
UNWIND $records AS rec
MATCH (c:Customer {segmentId:rec.segmentId})
SET c.segmentTitle = rec.title,
    c.segmentDescription = rec.description
""", records = [s.model_dump() for s in segment_descriptions])

## Create MCP Toolbox YAML
This will create the YAML configuration for an [MCP Toolbox server](https://neo4j.com/blog/developer/ai-agents-gen-ai-toolbox/). We will keep it simple by only including one tool for calculating potential churn risk.

Think of these as expert tools - pre-defined query templates with descriptions that can take in optional query parameters. We can later combine this with more generic MCP servers such as [mcp-neo4j-cypher](https://github.com/neo4j-contrib/mcp-neo4j/tree/main/servers/mcp-neo4j-cypher) that allow agents to write and execute their own Cypher queries based on the graph schema and user input.

In [ ]:
tool_yaml = f"""
sources:
    purchases-graph:
        kind: "neo4j"
        uri: "{NEO4J_URI}"
        user: "{NEO4J_USERNAME}"
        password: "{NEO4J_PASSWORD}"
tools:
  churn_risk_for_segments:
    kind: neo4j-cypher
    source: purchases-graph
     description: |
        Use this tool to get a list of customer segments with some amount of churn risk.  Customer segments are calculated beforehand based on common order patterns amung customers.  This function will return each segment with some churn risk, number of at risk customers, and estimated future lifetime value at risk of churning.
    statement: |
        MATCH (c:Customer)-[r:ORDERED]->(o:Order)
        WHERE NOT o:Return
        WITH c, max(o.invoiceDate) AS lastOrderDate, count(o) as totalOrders
        WHERE (lastOrderDate + duration({{months: 3}})) < dateTime("2025-08-31")
        WITH c
        MATCH (c)-[:ORDERED]->(o:Order)-[:LINE_ITEM]->(item:OrderLineItem)
        WHERE NOT o:Return
        WITH c, o,
          sum(item.quantity * item.unitPrice) AS orderRevenue
        WITH c,
          avg(orderRevenue) AS avgOrderRevenue,
          count(o) AS numberOfOrders,
          //for a real use case replace hard coded data with date()...demo data has a fixed date window
          duration.inDays(min(o.invoiceDate), date("2025-08-31")).days + 1 AS numberOfDays
        WHERE numberOfDays > 30
        WITH c,
          numberOfOrders,
          avgOrderRevenue,
          toFloat(numberOfOrders)/numberOfDays AS DailyOrderFrequency,
          365 AS averageCustomerLifespan

        RETURN c.segmentId AS segmentId,
          c.segmentTitle AS segmentTitle,
          c.segmentDescription AS segmentDescription,
          sum(avgOrderRevenue * DailyOrderFrequency * averageCustomerLifespan)AS churnRiskInDollars,
          count(c) AS numberOfAtRiskCustomers
        ORDER BY churnRiskInDollars DESC
"""
with open('tools-org.yaml', 'w') as file:
    file.write(tool_yaml)
print("tools.yaml file created successfully.")